In [ ]:
import os

RAP_PROJECT_ID = os.environ["DNANEXUS_PROJECT_ID"]  # set your own DNAnexus RAP project ID


In [ ]:
import yaml
import numpy as np
import polars as pl
import pandas as pd
from tqdm import tqdm
from statsmodels.stats.multitest import multipletests

from plotnine import *
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'

In [ ]:
RAP_ANNO_DIR = f"{RAP_PROJECT_ID}:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated"
LOCAL_ANNO_DIR = "PATH_TO_FILE"

# ANNO_FILE = "annotations_fillna_ukbgym_with_mane.parquet"
ANNO_FILE = "annotations_with_all_no_dup.parquet"

!dx download {RAP_ANNO_DIR}/{ANNO_FILE} -o {LOCAL_ANNO_DIR}/{ANNO_FILE}
anno = pl.scan_parquet(f"{LOCAL_ANNO_DIR}/{ANNO_FILE}")
anno.collect_schema().names()

diff_vars = (
    anno
    .filter(
        pl.col('ac_ukb')>=20, 
        pl.col('mac_ukb')<=20,
    )
    .collect(engine='streaming')
)

diff_vars

In [ ]:
diff_vars['loftee_hc'].value_counts()

In [ ]:
# Get gene trait associations
RAP_DIR = f'{RAP_PROJECT_ID}:/processed_data/REGENIE_results'

# ASSOC_FILE = 'loftee_mac20_associations_bh_corrected.parquet'
# ASSOC_FILE = 'regenie_127phenotypes_lofteeHC_mac20_EUR.parquet'
ASSOC_FILE = 'regenie_127phenotypes_lofteeHC_mac20_EUR_miss20per.parquet'

LOCAL_DIR = 'PATH_TO_FILE'

!dx download {RAP_DIR}/{ASSOC_FILE} -o {LOCAL_DIR}

gene_trait_df = pl.read_parquet(f'{LOCAL_DIR}/{ASSOC_FILE}')

gene_trait_df = (
    gene_trait_df
    .with_columns(
        pval_bh = pl.Series(
            multipletests(gene_trait_df['pval'].to_numpy(), method='fdr_bh')[1]
        ),
        pval_by = pl.Series(
            multipletests(gene_trait_df['pval'].to_numpy(), method='fdr_by')[1]
        )
    )
    # .filter(pl.col('pval_bh')<=0.05)
    .filter(pl.col('pval_by')<=0.05)
    .unique(subset=["region"], keep="first", maintain_order=True)
    # .select(['region', 'phenotype', 'pval_fdr'])
)

# CORR_FILE = "regenie_127phenotypes_lofteeHC_mac20_EUR_miss20per_correlations.parquet"
# !dx download {RAP_DIR}/{CORR_FILE} -o {LOCAL_DIR}

# loftee_corrs = (
#     pl.read_parquet(f'{LOCAL_DIR}/{CORR_FILE}')
#     .with_columns(
#         loftee_corr = pl.col('correlation'),
#         loftee_corr_abs = pl.col('correlation').abs(),
#         loftee_corr_dir = pl.col('correlation')/pl.col('correlation').abs(),
#     )
#     .select(['region', 'phenotype', 'loftee_corr', 'loftee_corr_abs', 'loftee_corr_dir']) 
# )

# gene_trait_df = (
#     gene_trait_df
#     .join(loftee_corrs, on=['region', 'phenotype'], how='inner')
#     .drop_nans()
#     .sort('loftee_corr_abs', descending=True)
#     .unique(subset=["region"], keep="first", maintain_order=True)
# )

gene_trait_df

In [ ]:
RVAT_DIR = f"{RAP_PROJECT_ID}:/processed_data/association_files"
LOCAL_DIR = "PATH_TO_FILE"

ac = 20

!dx download {RVAT_DIR}/loftee_hc_sum_ac{ac}_association_testing_results.parquet -o {LOCAL_DIR}
!dx download {RVAT_DIR}/am_pathogenicity_sum_ac{ac}_association_testing_results.parquet -o {LOCAL_DIR}

In [ ]:
lof = pl.read_parquet(f'{LOCAL_DIR}/loftee_hc_sum_ac{ac}_association_testing_results.parquet')
lof = (
    lof
    .with_columns(
        pval_bh = pl.Series(
            multipletests(lof['pval'].to_numpy(), method='fdr_bh')[1]
        ),
        pval_by = pl.Series(
            multipletests(lof['pval'].to_numpy(), method='fdr_by')[1]
        )
    )
    .filter((pl.col('pval_bh')<=0.05) | (pl.col('pval_by')<=0.05))
    # .filter(pl.col('pval_by')<=0.05)
    # .unique(subset=["gene"], keep="first", maintain_order=True)
)

lof

In [ ]:
lof_plot = (
    lof
    .with_columns(
        abs_beta = pl.col('beta').abs(),
        neg_log10_pval_bh = -pl.col('pval_bh').log10(),
        neg_log10_pval_by = -pl.col('pval_by').log10(),
        signif = (
            pl.when((pl.col('pval_by')<=0.05) & (pl.col('pval_bh')<=0.05))
            .then(pl.lit('BY and BH'))
            .otherwise(
                pl.when(pl.col('pval_by')<=0.05)
                .then(pl.lit('only BY'))
                .otherwise(
                    pl.when(pl.col('pval_bh')<=0.05)
                    .then(pl.lit('only BH'))
                    .otherwise(pl.lit('Not significant'))
                )
            )
        )
    )
)

# Calculate sample sizes for each group
n_stats = (
    lof_plot
    .group_by('signif')
    .agg(pl.count().alias('n'))
    .with_columns(
        signif_label = pl.concat_str([pl.col('signif'), pl.lit('\nn='), pl.col('n')])
    )
)

lof_plot_with_n = lof_plot.join(n_stats[['signif', 'signif_label']], on='signif')

(
    ggplot(lof_plot_with_n, aes(x='signif_label', y='abs_beta'))
    + geom_boxplot()
    + scale_y_log10()
    + theme_bw()
    + theme(
        figure_size=(2.5, 4),
    )
    + labs(x='Significance Category')
)

In [ ]:
am = pl.read_parquet(f'{LOCAL_DIR}/am_pathogenicity_sum_ac{ac}_association_testing_results.parquet')

am = (
    am
    .with_columns(
        pval_bh = pl.Series(
            multipletests(am['pval'].to_numpy(), method='fdr_bh')[1]
        ),
        pval_by = pl.Series(
            multipletests(am['pval'].to_numpy(), method='fdr_by')[1]
        )
    )
    .filter((pl.col('pval_bh')<=0.05) | (pl.col('pval_by')<=0.05))
    # .filter(pl.col('pval_by')<=0.05)
    # .unique(subset=["gene"], keep="first", maintain_order=True)
)

am

In [ ]:
am_plot = (
    am
    .with_columns(
        abs_beta = pl.col('beta').abs(),
        neg_log10_pval_bh = -pl.col('pval_bh').log10(),
        neg_log10_pval_by = -pl.col('pval_by').log10(),
        signif = (
            pl.when((pl.col('pval_by')<=0.05) & (pl.col('pval_bh')<=0.05))
            .then(pl.lit('BY and BH'))
            .otherwise(
                pl.when(pl.col('pval_by')<=0.05)
                .then(pl.lit('only BY'))
                .otherwise(
                    pl.when(pl.col('pval_bh')<=0.05)
                    .then(pl.lit('only BH'))
                    .otherwise(pl.lit('Not significant'))
                )
            )
        )
    )
)

# Calculate sample sizes for each group
n_stats = (
    am_plot
    .group_by('signif')
    .agg(pl.count().alias('n'))
    .with_columns(
        signif_label = pl.concat_str([pl.col('signif'), pl.lit('\nn='), pl.col('n')])
    )
)

am_plot_with_n = am_plot.join(n_stats[['signif', 'signif_label']], on='signif')

(
    ggplot(am_plot_with_n, aes(x='signif_label', y='abs_beta'))
    + geom_boxplot()
    + scale_y_log10()
    + theme_bw()
    + theme(
        figure_size=(2.5, 4),
    )
    + labs(x='Significance Category')
)